# 01 · Full Predictive Distribution (PDF / CDF)

**The richest form of probabilistic forecast**: for each future month we produce a complete probability distribution
$\hat F_{T+h|T}$. Every other type (quantiles, intervals, event probabilities, categories) can be *derived* from it.

## Theory

### How models produce a full distribution
1. **Parametric / model-based**: assume a data-generating process with a noise distribution.
   ARIMA and ETS (state-space) models with Gaussian innovations give Gaussian predictive distributions:
   $$Y_{T+h}\mid\mathcal I_T \sim N\!\left(\hat y_{T+h|T},\ \sigma_h^2\right)$$
2. **Distributional regression**: predict *all parameters* of a distribution from features,
   e.g. GAMLSS, NGBoost, DeepAR (which outputs $\mu_t,\sigma_t$ or NegBin parameters per step).
3. **Non-parametric densities**: mixture density networks, kernel density on residuals, normalising flows.

### Why the variance grows with the horizon
Write ARIMA in MA($\infty$) form $y_t = \sum_{j\ge 0}\psi_j\varepsilon_{t-j}$. The $h$-step forecast error is
$e_{T+h}=\sum_{j=0}^{h-1}\psi_j\varepsilon_{T+h-j}$, so
$$\sigma_h^2=\sigma^2\sum_{j=0}^{h-1}\psi_j^2 .$$
For a random walk, $\psi_j=1$ so $\sigma_h=\sigma\sqrt h$. Errors accumulate. (This *ignores parameter
uncertainty*, so it is slightly optimistic. See notebooks 05 and 11.)

### Transformations
We model $\log y$ (see nb 00). A Gaussian on the log scale becomes a **log-normal** distribution on the original scale:
$$Y\sim\text{LogNormal}(\mu,\sigma):\quad f_Y(y)=\frac{1}{y\,\sigma\sqrt{2\pi}}\exp\!\left(-\frac{(\log y-\mu)^2}{2\sigma^2}\right).$$
The $1/y$ is the **Jacobian**. Remember it when computing log scores on the original scale.
Right-skewness appears automatically, so upside uncertainty is larger than downside.

### Evaluating a full distribution
* **Log score** $-\log\hat f(y)$: local, strictly proper, very sensitive to tails.
* **CRPS**: integrates over the whole CDF, robust, in the units of $y$.
* **PIT histogram**: calibration check.

**Model used:** the "airline model" SARIMA$(0,1,1)(0,1,1)_{12}$ on $\log y$.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(1)

air = pf.load_air_passengers()
logy = np.log(air)
train, test = air[:"1959-12-01"], air["1960-01-01":]
H = len(test)

## 1. Fit the model and get the predictive distribution for 1960

In [2]:
mod = SARIMAX(np.log(train), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12))
res = mod.fit(disp=False)
print(res.summary().tables[1])

fc = res.get_forecast(steps=H)
mu = np.asarray(fc.predicted_mean)              # mean of log y
sd = np.sqrt(np.asarray(fc.var_pred_mean))       # std of log y
dist = stats.lognorm(s=sd, scale=np.exp(mu))     # vectorised log-normal on the original scale

pd.DataFrame({"mu (log)": mu, "sigma_h (log)": sd, "median": dist.median(), "mean": dist.mean(),
              "P05": dist.ppf(0.05), "P95": dist.ppf(0.95), "actual": test.values}, index=test.index)

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1         -0.3483      0.081     -4.299      0.000      -0.507      -0.189
ma.S.L12      -0.5620      0.094     -5.953      0.000      -0.747      -0.377
sigma2         0.0013      0.000      8.495      0.000       0.001       0.002


,mu (log),sigma_h (log),median,mean,P05,P95,actual
1960-01-01,6.039,0.036,419.329,419.604,395.072,445.075,417.0
1960-02-01,5.989,0.043,398.921,399.294,371.533,428.328,391.0
1960-03-01,6.145,0.049,466.581,467.147,430.262,505.965,419.0
1960-04-01,6.119,0.055,454.408,455.087,415.355,497.133,461.0
1960-05-01,6.160,0.060,473.270,474.109,429.135,521.944,472.0
1960-06-01,6.305,0.064,547.125,548.247,492.434,607.889,535.0
1960-07-01,6.433,0.068,622.229,623.679,556.162,696.144,622.0
1960-08-01,6.446,0.072,630.167,631.812,559.593,709.643,606.0
1960-09-01,6.267,0.076,526.752,528.274,464.874,596.866,508.0
1960-10-01,6.136,0.080,462.295,463.760,405.591,526.927,461.0


Notice: **mean > median** (log-normal skew), and $\sigma_h$ **increases with horizon**.

## 2. Visualise the densities: the forecast is a whole distribution per month

In [3]:
taus = [0.025, 0.1, 0.25, 0.5, 0.75, 0.9, 0.975]
fig = pf.add_fan(go.Figure(), test.index, {t: dist.ppf(t) for t in taus}, history=train, actual=test.values, fmt=",.0f")
fig.update_layout(title="Fan chart: quantiles of the predictive distribution (hover a month to read every interval)",
                  yaxis_title="passengers (thousands)")
fig.show()

grid = np.linspace(300, 750, 600)
fig = go.Figure()
for k, h in enumerate([1, 6, 12]):
    d = stats.lognorm(s=sd[h - 1], scale=np.exp(mu[h - 1]))
    lab = f"h={h} ({test.index[h-1]:%b %Y})"
    fig.add_trace(go.Scatter(x=grid, y=d.pdf(grid), name=lab, line=dict(color=pf.SERIES[k], width=2),
                             hovertemplate=lab + ": density %{y:.4f} at %{x:.0f}<extra></extra>"))
    pf.add_vline(fig, test.iloc[h - 1], 0, d.pdf(grid).max(), f"actual {test.index[h-1]:%b %Y}", color=pf.SERIES[k], dash="dot")
fig.update_layout(title="Predictive densities widen and skew right as the horizon grows (dotted = actual)",
                  xaxis_title="passengers (thousands)", yaxis_title="density")
fig.show()

## 3. Uncertainty growth with horizon: $\sigma_h^2=\sigma^2\sum\psi_j^2$

Let's verify the statsmodels forecast variance against the $\psi$-weight formula.

In [4]:
from statsmodels.tsa.arima_process import arma2ma
theta, Theta, sigma2 = res.params["ma.L1"], res.params["ma.S.L12"], res.params["sigma2"]
# (1-B)(1-B^12) y_t = (1+theta B)(1+Theta B^12) e_t  -> expand the polynomials
ar_poly = np.polymul([1, -1], np.r_[1, np.zeros(11), -1])
ma_poly = np.polymul([1, theta], np.r_[1, np.zeros(11), Theta])
psi = arma2ma(ar_poly, ma_poly, lags=H)
var_formula = sigma2 * np.cumsum(psi**2)

hh = np.arange(1, H + 1)
fig = go.Figure()
fig.add_trace(go.Scatter(x=hh, y=sd, mode="lines+markers", name="statsmodels σ_h", line=dict(color=pf.SERIES[0], width=2),
                         marker=dict(size=10)))
fig.add_trace(go.Scatter(x=hh, y=np.sqrt(var_formula), mode="markers", name="σ·√Σψ² (formula)",
                         marker=dict(symbol="x", size=9, color=pf.SERIES[1])))
fig.update_layout(title="Forecast uncertainty grows with the horizon: the two calculations agree", height=360,
                  hovermode="x unified", xaxis_title="horizon h (months)", yaxis_title="std of log y")
fig.show()

## 4. Anything can be derived from the full distribution

Because we have $\hat F$, we can answer any question, without refitting:

In [5]:
july = test.index.get_loc(pd.Timestamp("1960-07-01"))
d = stats.lognorm(s=sd[july], scale=np.exp(mu[july]))
print(f"July 1960:  P(passengers > 600) = {d.sf(600):.3f}")
print(f"            90% central interval = [{d.ppf(0.05):.0f}, {d.ppf(0.95):.0f}]")
print(f"            expected value given > P90 (tail mean) = {d.expect(lambda x: x, lb=d.ppf(0.9)) / 0.1:.0f}")

# Annual total for 1960 needs the JOINT distribution of the 12 months (see notebook 05)
# -> simulate sample paths from the state-space model:
paths = np.exp(np.asarray(res.simulate(nsimulations=H, repetitions=4000, anchor="end")).reshape(H, -1))
annual = paths.sum(axis=0)
print(f"Annual 1960 total: median {np.median(annual):.0f}, 90% PI [{np.quantile(annual,.05):.0f}, {np.quantile(annual,.95):.0f}], actual {test.sum():.0f}")

July 1960:  P(passengers > 600) = 0.703
            90% central interval = [556, 696]
            expected value given > P90 (tail mean) = 702


Annual 1960 total: median 5861, 90% PI [5343, 6434], actual 5714


## 5. Proper evaluation: many forecasts, not 12

Scoring a distribution on only 12 points is extremely noisy. We create **one-step-ahead forecasts for 1955–1960
(72 months)** with parameters estimated on 1949–1954 only (no look-ahead), using the Kalman filter.

Three competitors:
* **A. SARIMA on log y → log-normal**
* **B. SARIMA on raw y → Gaussian** (ignores multiplicative variance)
* **C. Seasonal naive + drift, Gaussian**: $\hat y_t=y_{t-12}+\bar d$ with $\sigma$ from training seasonal differences

In [6]:
cut = "1954-12-01"
start = "1955-01-01"
y_eval = air[start:]

def one_step(series, params_from):
    m_tr = SARIMAX(series[:cut], order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    r_all = SARIMAX(series, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).filter(m_tr.params)
    p = r_all.get_prediction(start=start)
    return np.asarray(p.predicted_mean), np.sqrt(np.asarray(p.var_pred_mean))

muA, sdA = one_step(logy, cut)
muB, sdB = one_step(air, cut)
d12 = (air - air.shift(12))[:cut].dropna()
muC = air.shift(12)[start:].to_numpy() + d12.mean()
sdC = np.full_like(muC, d12.std())

yv = y_eval.to_numpy()
samplesA = np.exp(rng.normal(muA, sdA, size=(4000, len(yv))))
scores = pd.DataFrame({
    "CRPS": [pf.crps_sample(yv, samplesA).mean(), pf.crps_gaussian(yv, muB, sdB).mean(), pf.crps_gaussian(yv, muC, sdC).mean()],
    "log score": [np.mean(-(stats.norm.logpdf(np.log(yv), muA, sdA) - np.log(yv))),   # Jacobian!
                  pf.log_score_gaussian(yv, muB, sdB).mean(), pf.log_score_gaussian(yv, muC, sdC).mean()],
    "90% coverage": [pf.coverage(yv, np.exp(stats.norm.ppf(.05, muA, sdA)), np.exp(stats.norm.ppf(.95, muA, sdA))),
                     pf.coverage(yv, stats.norm.ppf(.05, muB, sdB), stats.norm.ppf(.95, muB, sdB)),
                     pf.coverage(yv, stats.norm.ppf(.05, muC, sdC), stats.norm.ppf(.95, muC, sdC))],
}, index=["A: SARIMA log → lognormal", "B: SARIMA raw → Gaussian", "C: seasonal naive+drift"])
scores

,CRPS,log score,90% coverage
A: SARIMA log → lognormal,6.888,3.940,0.958
B: SARIMA raw → Gaussian,8.007,4.288,0.750
C: seasonal naive+drift,15.718,5.225,0.458


In [7]:
pits = {"A: lognormal": stats.norm.cdf(np.log(yv), muA, sdA),
        "B: Gaussian raw": stats.norm.cdf(yv, muB, sdB),
        "C: snaive+drift": stats.norm.cdf(yv, muC, sdC)}
fig = pf.subplots(1, 3, titles=list(pits), shared_yaxes=True, height=330, title="PIT of the one-step forecasts, 1955–1960")
for c, u in enumerate(pits.values(), start=1):
    pf.add_pit(fig, u, bins=8, row=1, col=c)
fig.update_layout(margin=dict(t=100))
fig.show()

# Heteroscedasticity check: the raw-scale model has constant sigma, but errors grow with the level
fig = go.Figure()
for k, (lab, e) in enumerate([("A: log-normal (errors on log scale)", (np.log(yv) - muA) / sdA),
                              ("B: Gaussian on raw y", (yv - muB) / sdB)]):
    fig.add_trace(go.Scatter(x=y_eval.index, y=e, mode="lines+markers", name=lab, line=dict(color=pf.SERIES[k], width=1.5),
                             marker=dict(size=8)))
fig.add_hline(y=0, line=dict(color=pf.INK["axis"], width=1))
fig.update_layout(title="Standardised one-step errors", height=360, hovermode="x unified", yaxis_title="error / σ")
fig.show()

## 6. Takeaways

* A full predictive distribution is the **most informative** forecast. Quantiles, intervals, exceedance probabilities,
  tail expectations and (with paths) aggregates are all derived from it.
* The distributional **assumption** matters: modelling on the log scale gives a skewed, level-dependent spread that
  matches the data much better than a constant-variance Gaussian.
* Score with CRPS and the log score on **many** rolling forecasts; check calibration with PIT.
* Limitations of the parametric Gaussian route: wrong tails if innovations are heavy-tailed (try Student-t),
  no parameter uncertainty (nb 11), and marginal distributions only (nb 05, 09 for joint).

### Exercises
1. Replace the Gaussian with a Student-t on log residuals: does the log score improve?
2. Compare the lognormal **mean** vs **median** as point forecasts using MSE and MAE. Which wins on each? (nb 00 §7)
3. Fit ETS(M,A,M) with `statsmodels.tsa.exponential_smoothing.ets.ETSModel` and compare CRPS.